# Rapport de projet PE_V2X_Reliability (Approche A)

Ce rapport présente de manière systématique : la motivation de recherche, la modélisation à trois scénarios, les mécanismes et la définition des métriques, la conception de la reproductibilité, ainsi que la décomposition et la discussion des résultats Fig01–Fig07. Les figures Fig01–Fig07 sont présentées dans le corps du texte sous forme d’aperçus PNG (rendus à partir du PDF de référence).

**Date:** 2026-03-01


## 1 Motivation de recherche et définition du problème

L’exigence centrale des messages de sécurité V2X n’est pas « le débit maximal » ni « la latence moyenne minimale ». Il s’agit plutôt de satisfaire durablement, sous **topologie hautement dynamique**, **environnements de propagation complexes** et **contention sur un canal radio partagé**, les **contraintes de fiabilité et de temporalité** requises par la perception et la décision coopératives. Sur route, les environnements dégradés (masquage urbain, tunnels) et la congestion en heures de pointe ne sont pas des « cas extrêmes » rares ; ce sont des conditions limites courantes auxquelles un système doit faire face : rues canyon, échangeurs et longs tunnels sont répandus, et les pics de trafic augmentent fréquemment le taux d’occupation du canal. Construire un cadre d’évaluation **explicable, reproductible et extensible** autour de la « fiabilité et temporalité V2X en environnement dégradé » a donc une valeur d’ingénierie et de recherche claire.

Du point de vue applicatif, les modes d’échec des messages de sécurité V2X doivent au moins être distingués en deux catégories, car leurs impacts sur les applications de sûreté suivent des voies différentes :

1) **Échec de fiabilité (Non-delivery)** : le paquet n’arrive pas, ce qui se traduit par une baisse du PDR (Packet Delivery Ratio).  
2) **Échec de temporalité (Late delivery)** : le paquet arrive mais dépasse l’échéance (deadline) ; pour la décision coopérative il est équivalent à un échec, i.e., un « échec implicite ».

Cela implique qu’un indicateur unique (p. ex. la latence moyenne ou un PDR unique) ne suffit pas pour conclure rigoureusement : les moyennes masquent souvent le **risque de queue (tail latency)** ; et en présence d’une deadline, un déplacement vers la droite de la distribution de queue se convertit directement en paquets tardifs, réduisant le taux de succès effectif de l’application. Afin d’éviter des « constats sans attribution causale », ce projet adopte une modélisation système **mécanistiquement interprétable**, et rapporte sous un protocole unifié :

- **Décroissance de fiabilité avec la distance** : PDR vs distance ;  
- **Distribution de délai et risque de queue** : CDF du délai, p95/p99 ;  
- **Succès effectif tenant compte de la deadline** : `success_phy` (succès PHY), `late` (dépassement), `success` (succès à temps).

Par ailleurs, l’environnement dégradé est consolidé en trois scénarios représentatifs (RefPlus / UrbMaskPlus / TunnelPlus), et un contraste à deux régimes **NoCong / Cong** est introduit. En gelant le reste du protocole, les effets de congestion (contention et collisions) sont injectés uniquement via un interrupteur de congestion, afin de répondre à deux questions proches des « décisions de conception système » :

- La congestion en heures de pointe devient-elle le facteur dominant commun, comprimant les différences d’environnement et modifiant les choix de stratégie ?  
- Avec les retransmissions `ret=0/1/2` comme levier principal, comment arbitrer entre bénéfices (hausse de fiabilité) et coûts (queue de délai plus épaisse, hausse du taux de late) selon les scénarios/régimes ?

---

## 2 Contenu de construction et trajectoire globale

Les résultats du projet se résument en une **chaîne d’évaluation système** : génération des entrées → simulation événementielle → statistiques agrégées → production de figures, figée sous forme de paquet reproductible via **contrat de sortie run_id + instantanés de commandes**. L’objectif n’est pas de « produire un résultat unique », mais de bâtir un squelette réutilisable permettant, lors d’extensions futures (autres environnements dégradés, RSU, stratégies améliorées, trajectoires réelles), de conserver la même définition des métriques et la même chaîne de preuves.

### 2.1 Génération des entrées : rendre le scénario et le trafic explicites et auditables

Pour éviter que des réglages critiques soient implicites dans le code (et donc non reproductibles), les entrées sont décomposées et figées en fichiers/configurations traçables :

- **Mobilité (trajectoires)** : géométrie RefPlus + suivi IDM + phases de feux + mécanismes cross/turn, générant un CSV de trajectoires.  
- **Bâtiments (UrbMask)** : géométrie de quartier exprimée par des blocs rectangulaires, générant un CSV de bâtiments (empreinte, hauteur, motif de distribution, etc.).  
- **Tunnel (Tunnel)** : intervalle de tunnel et paramètres de dégradation, générant un CSV de configuration tunnel.

La valeur d’ingénierie est que l’évaluateur n’a pas besoin de l’arborescence locale de l’auteur ni de comprendre chaque ligne de code ; il peut reproduire les conditions à partir des fichiers d’entrée et des instantanés de commandes.

### 2.2 Simulation événementielle : l’échantillon paquet comme unité de base

Le cœur de simulation traite chaque message de sécurité comme un échantillon et produit trois types d’informations :

- **Variables géométrie/état** : distance, position du milieu de lien, état de scénario (p. ex. NLOS / dans le tunnel), etc.  
- **Variables de résultat** : `success_phy`, `delay_ms`, `late`, `success`.  
- **Champs de preuve** : intensité de masquage (p. ex. `d_min`/`b`), position dans le tunnel (`tunnel_u`), proxys de congestion (`n_cs`/CBR/`p_col`/`cong_delay_ms`), etc.

Ces champs de preuve permettent ensuite non seulement d’observer « ça se dégrade », mais aussi d’expliquer « pourquoi ».

### 2.3 Agrégation : compresser le raw en trois tables adaptées à la relecture

Comme le raw paquet peut être volumineux, la vérification s’aligne sur des tables CSV agrégées :

- **summary_metrics** : PDR par bins de distance, p50/p95/p99, late ratio, plus moyennes des champs de preuve ;  
- **position_heterogeneity** : table d’hétérogénéité UrbMask à distance fixe (dans une bande de distance, binning par `mid_x`) ;  
- **tunnel_segments** : table de statistiques segmentées Tunnel (dans une bande de distance, binning par `tunnel_u`).

Ce système à trois tables n’est pas « pour multiplier les fichiers » ; il correspond strictement aux explications mécanistes : Fig05 nécessite la table d’hétérogénéité ; Fig06 nécessite la table segmentée—sinon la conclusion n’est pas auditables.

### 2.4 Production de figures : distinguer « figures finales » et « figures de transparence »

Deux catégories de figures sont produites :

- **Figures finales (Fig01–Fig07)** : définitions unifiées, forte densité d’information, destinées à la soumission/soutenance ;  
- **Figures de soutien (deliverable F1/F2/F3)** : couvrent toutes les combinaisons (scénario × ret × seeds) pour transparence et exploration ; elles ne remplacent pas les définitions finales.

---

## 3 Signification, valeur et contributions

Cette section adopte une formulation académique « valeur—contribution—vérifiabilité », cohérente avec le travail réellement réalisé (sans exagération).

### 3.1 Valeur : pourquoi ce travail est important et nécessaire

**(1) Définitions alignées sur l’application : expliciter “late” dans la définition du succès**  
Dans un système réel, « reçu mais trop tard » équivaut à un échec pour la décision coopérative. Le projet distingue explicitement `success_phy` et `success`, et utilise `late` pour représenter les échecs de temporalité, afin que les conclusions soient directement exploitables au regard des contraintes applicatives.

**(2) Mécaniser des phénomènes clés : quantifier l’hétérogénéité et la segmentation**  
La difficulté principale des rues canyon n’est pas seulement « une moyenne plus mauvaise », mais l’hétérogénéité à distance égale ; la difficulté principale des tunnels n’est pas seulement « globalement plus mauvais », mais la dégradation segmentée et l’épaississement de la queue. Via `position_heterogeneity` et `tunnel_segments`, le projet transforme ces intuitions en résultats statistiques vérifiables.

**(3) Arbitrages explicables sous congestion et stratégies : éviter les conclusions à indicateur unique**  
Les gains de retransmission s’accompagnent souvent de coûts de queue ; sous congestion, l’amélioration PHY peut être compensée par une hausse des late. Avec la décomposition (Fig03) et les preuves proxy (Fig04), l’arbitrage est ancré dans des champs de données et des figures.

**(4) Reproductibilité forte : vérification et re-tracé sur toute machine**  
Contrat run_id, instantanés de commandes, tables agrégées et figures finales permettent une vérification indépendante de l’environnement de l’auteur, tout en conservant un squelette stable pour des extensions futures.

### 3.2 Contributions : contributions concrètes du projet

**Contribution 1 : construction normalisée à trois scénarios et modélisation interprétable**  
- RefPlus : base contrôlable (géométrie + trafic + feux) pour l’attribution ;  
- UrbMaskPlus : intensité continue `d_min→b` + mélange LOS/NLOS + terme de réflexion équivalent optionnel, capturant l’hétérogénéité à distance égale ;  
- TunnelPlus : position segmentée `tunnel_u` + courbes bell+fade + injection de délai de queue, capturant segmentation et effets de queue.

**Contribution 2 : chaîne de preuves des proxys de congestion et cadre d’ablation Cong-only**  
- `n_cs`/CBR/`p_col`/`cong_delay_ms` rendent explicite la chaîne « densité → canal occupé → collisions/attente → queue plus épaisse » ;  
- NoCong vs Cong ne diffèrent que par un interrupteur, garantissant l’attribution.

**Contribution 3 : évaluation gain–coût unifiée avec ret=0/1/2 comme levier principal**  
- Montrer non seulement le gain PDR, mais aussi l’élévation p95/p99 et le risque late ;  
- Passer de « maximiser le PDR » à « compromis sous contraintes conjointes fiabilité & temporalité ».

**Contribution 4 : conception de chaîne de preuves auditables (figure–table–champ)**  
- Fig05 ⇔ position_heterogeneity ⇔ (`d_min`, `b`, `g_refl`) ;  
- Fig06 ⇔ tunnel_segments ⇔ (`tunnel_u`, queue de délai) ;  
- Fig03/Fig04 ⇔ (`success_phy`, `late`, `p_col`, `cong_delay_ms`).

### 3.3 Vérifiabilité : chemin minimal de vérification

Chemin recommandé :

1) Parcourir Fig01–Fig07 (boucle de conclusion) ;  
2) Vérifier run_commands (protocole gelé, différence minimale NoCong/Cong) ;  
3) Contrôler les champs et définitions des tables (tables) ;  
4) En cas de besoin, consulter F1/F2/F3 et les tables associées.

---

## 4 Construction des trois scénarios (détaillée, avec système d’équations)

Ce chapitre suit une écriture académique « modèle système / réglages de scénario » : conventions communes puis détails RefPlus/UrbMaskPlus/TunnelPlus, avec un jeu d’équations directement réutilisable. Ce projet vise une évaluation système à complexité contrôlée pour obtenir des conclusions interprétables, sans se substituer à un lancer de rayons complet ni à une simulation protocolaire full-stack.

### 4.0 Conventions : coordonnées, temps discret et milieu de lien

- Coordonnées planes : couloir principal selon \(x\), décalage latéral selon \(y\).  
- Simulation en temps discret de pas \(\Delta t\) ; messages générés à la fréquence \(f_m\).  
- Pour un lien Tx/Rx :

$$
d=\|\mathbf{p}_{tx}-\mathbf{p}_{rx}\|_2,\qquad 
\mathbf{p}_{mid}=\frac{\mathbf{p}_{tx}+\mathbf{p}_{rx}}{2},\qquad
x_{mid}=\mathbf{p}_{mid,x}.
$$

### 4.1 RefPlus : base contrôlable (géométrie + IDM + feux + cross/turn)

#### 4.1.1 Géométrie : ligne centrale et multi-voies

RefPlus utilise un couloir de 3 km ; la courbe en S est décrite par une fonction lisse (exemple) :

$$
y_c(x)=A\sin\!\Big(2\pi\,\frac{x-x_0}{x_1-x_0}\Big)\cdot \mathbb{I}_{[x_0,x_1]}(x),
$$

où \(A\) est l’amplitude latérale et \([x_0,x_1]\) l’intervalle de courbe. Les centres de voies :

$$
y_{\ell}^{(\pm)}(x)=y_c(x)\ \pm\ \Big(\frac{g}{2}+\big(\ell+\tfrac{1}{2}\big)w\Big),
$$

où \(g\) est la largeur du séparateur central, \(w\) la largeur de voie, et \(\ell\in\{0,1,2\}\) indexe les 3 voies par sens.

#### 4.1.2 Trafic : modèle IDM (files / pelotons / ondes stop-and-go)

$$
\dot v = a\Big[1-\Big(\frac{v}{v_0}\Big)^{\delta}-\Big(\frac{s^{\ast}(v,\Delta v)}{s}\Big)^2\Big],
\qquad
s^{\ast}(v,\Delta v)=s_0+vT+\frac{v\Delta v}{2\sqrt{ab}},
$$

où \(v\) est la vitesse, \(\Delta v\) la vitesse relative, \(s\) l’écart, \(v_0\) la vitesse désirée, \(T\) le temps de tête, \(a,b\) l’accélération max / décélération confortable, \(s_0\) l’écart minimal, et \(\delta\) est typiquement 4. Sous contrainte de feux, le modèle génère naturellement files et libérations de pelotons, fournissant une origine structurée au proxy \(n_{cs}\).

#### 4.1.3 Feux et carrefours : phases et offsets

Avec période \(C\) et vert principal \(G\), l’indicateur de passage :

$$
\phi_{main}(t)=\mathbb{I}\big(\ (t+\tau)\bmod C\in[0,G)\ \big),
$$

où \(\tau\) est l’offset. Dans la zone de carrefour, un flux transversal et des probabilités de tournants (droite/gauche/tout droit) contrôlent l’entrelacement et les pics de densité locale.

### 4.2 UrbMaskPlus : masquage urbain (bâtiments + intensité continue + réflexion équivalente)

#### 4.2.1 Géométrie bâtiments : ensemble de rectangles

Les empreintes sont un ensemble \(\mathcal{B}=\{B_k\}\), chaque \(B_k\) défini par \((x_{min},x_{max},y_{min},y_{max},h)\).

#### 4.2.2 Distance minimale segment–bâtiment \(d_{min}\)

$$
d_{min}=\min_{B_k\in\mathcal{B}} \ \mathrm{dist}\big(\overline{\mathbf{p}_{tx}\mathbf{p}_{rx}},\partial B_k\big)
$$

#### 4.2.3 Intensité de masquage \(b\)

$$
b=\exp\!\left(-\Big(\frac{d_{min}}{T}\Big)^2\right)
$$

#### 4.2.4 Succès mixte LOS/NLOS

$$
p_{succ}(d,b)=(1-b)\,p_{LOS}(d)+b\,p_{NLOS}(d)
$$

#### 4.2.5 Terme de réflexion équivalent (optionnel)

$$
G_{refl}(d_{min},b)=G_{max}\exp(-d_{min}/d_0)\,b
$$

#### 4.2.6 Hétérogénéité à distance fixe (binning par \(x_{mid}\))

$$
\mathcal{S}_j=\{\,\text{links}: d\in[d_1,d_2],\ x_{mid}\in I_j\,\}
$$

### 4.3 TunnelPlus : dégradation segmentée (tunnel_u + bell+fade + injection de queue)

#### 4.3.1 Position normalisée \(u\)

$$
u=\mathrm{clip}\Big(\frac{x_{mid}-x_0}{x_1-x_0},0,1\Big)
$$

#### 4.3.2 Forme : bell + fade

$$
\mathrm{bell}(u)=\sin^2(\pi u),\qquad \mathrm{bell}_\gamma(u)=\mathrm{bell}(u)^\gamma
$$

#### 4.3.3 Injection de délai de queue (expression indicative)

$$
D = D_0 + b_{tun}\cdot D_{extra},\qquad D_{extra}\sim \mathrm{Exp}(\lambda)
$$

#### 4.3.4 Statistiques segmentées (binning par \(u\))

$$
\mathcal{T}_j=\{\,\text{links}: d\in[d_1,d_2],\ u\in J_j\,\}
$$

### 4.4 Champs de preuve (fermeture “phénomène–mécanisme–implication”)

- UrbMask : `d_min`, `b`, `g_refl` ⇔ hétérogénéité (Fig05)  
- Tunnel : `tunnel_u` ⇔ segmentation (Fig06)  
- Cong : `n_cs`, CBR, `p_col`, `cong_delay_ms` ⇔ décomposition des échecs (Fig03/04)

---

## 5 Modélisation mécaniste et système de métriques

Définir \(S_{phy}\in\{0,1\}\), \(L\in\{0,1\}\), \(S=S_{phy}(1-L)\), et utiliser :

$$
\mathrm{PDR}\approx \mathrm{PDR}_{phy}\cdot (1-\mathrm{LateRatio}_{phy})
$$

Expression conceptuelle retransmission + délai :

$$
\mathbb{P}(S_{phy}=1)=1-\prod_{k=0}^{ret}(1-p_k),
\qquad
D = D_{base} + D_{prop} + D_{queue} + D_{cong} + D_{retry}
$$

Proxys de congestion (expression indicative) :

$$
\mathrm{CBR}=\min\big(1,\ (n_{cs}-1)\lambda\tau_{air}\big),
\qquad
p_{col}=1-\exp(-k\,\mathrm{CBR}),
\qquad
D_{cong}\sim \mathrm{Exp}(\beta(\mathrm{CBR}))
$$

---

## 6 Reproductibilité et audit (formulation rigoureuse)

On note `$ROOT` la racine du dépôt ; la sortie suit `runs/<run_id>/` et inclut `run_commands.txt`. La différence NoCong/Cong est limitée à l’interrupteur de congestion pour garantir l’attribution. Reproduction recommandée : smoke → small → S20.


# Décomposition et discussion des résultats (Raw-derived, S20)

## 1. Sources de données et chaîne statistique (raw → cache → figures)

Ces résultats proviennent de deux expériences finales en contrôle strict : **A_Final_NoCong_S20** et **A_Final_Cong_S20**. La configuration commune (définie par les commandes finales et les fichiers de configuration) est identique : scénarios **Ref / UrbMask / Tunnel**, stratégies **ret=0/1/2**, seeds **seed=1–20**, et statistiques centrées sur **0–200 m**. La seule différence est l’activation de la compétition/congestion, constituant une ablation stricte (NoCong vs Cong).

Pour éviter des courbes « peu lisses / peu crédibles » dues à des tables trop clairsemées, les figures ne sont pas générées directement depuis les tables. Une chaîne en deux étapes est utilisée :
(1) agrégation du raw par bins de distance afin de collecter comptages et histogrammes nécessaires aux quantiles ;  
(2) mise en cache des résultats agrégés en `.mat` (reproductibilité et re-tracé rapide), puis rendu uniforme des 7 figures finales sous MATLAB.
Les noms de cache rendent visibles les résolutions clés, p. ex. **distance bin=2 m**, **delay histogram bin=0.25 ms**, et la bande d’analyse mécaniste **band=80–100 m** (hétérogénéité spatiale UrbMask et segmentation Tunnel).

---

## 2. Définition des métriques et niveaux de “succès” (éviter l’ambiguïté)

En présence de contraintes de deadline, le « succès » en V2X comporte au moins deux niveaux : **succès PHY** et **succès à temps**. Le rapport utilise donc trois métriques complémentaires :

1. **timely PDR (noté PDR ou timely_rate dans les figures)**  
$$
\mathrm{PDR}*{\mathrm{timely}}(d)=\frac{N*{\mathrm{success}}(d)}{N_{\mathrm{total}}(d)}
$$
où (N_{\mathrm{success}}) est le nombre de paquets arrivés avant la deadline.

2. **PHY success rate (noté phy_rate)**  
$$
\mathrm{PDR}*{\mathrm{phy}}(d)=\frac{N*{\mathrm{success_phy}}(d)}{N_{\mathrm{total}}(d)}
$$
où (N_{\mathrm{success_phy}}) est le nombre de paquets décodés avec succès au PHY (sans exigence de temporalité).

3. **late ratio parmi les succès PHY (noté late_ratio_phy)**  
$$
\mathrm{late_ratio}*{\mathrm{phy}}(d)=\frac{N*{\mathrm{late}}(d)}{N_{\mathrm{success_phy}}(d)}
$$
où (N_{\mathrm{late}}) désigne les paquets « succès PHY mais hors deadline ».

Ces métriques satisfont l’identité clé (Fig03) :
$$
\mathrm{PDR}*{\mathrm{timely}}(d)=\mathrm{PDR}*{\mathrm{phy}}(d)\cdot\left(1-\mathrm{late_ratio}_{\mathrm{phy}}(d)\right)
$$
Elle décompose la baisse du « timely PDR » en : **(i) baisse du succès PHY** et **(ii) hausse de la pénalité late parmi les succès PHY**, ce qui soutient une argumentation fermée « phénomène–mécanisme–implication ».

---

# Fig01–Fig07 : déconstruction figure par figure (réutilisable comme texte + légendes)

## Fig01 : PDR vs distance (dist≤200 m ; NoCong plein, Cong pointillé)

![](assets/final_figures_A_preview/Fig01_PDR_Focus.png)

**Sens et rôle**  
Fig01 est la figure principale, répondant : en environnement dégradé et sous congestion, comment la fiabilité (timely PDR) se dégrade avec la distance ? Quel gain apporte `ret` ?

**Observation 1 : NoCong — dégradation dominée par la propagation et gain monotone de ret**  
Sous NoCong, les courbes décroissent avec la distance ; `ret` de 0→1→2 décale les courbes vers le haut. Fig07 fournit des ancrages (dist≤200 m) :

- Ref : PDR = 0.439 (ret0) → 0.537 (ret1) → 0.582 (ret2)  
- UrbMask : PDR = 0.462 (ret0) → 0.584 (ret1) → 0.644 (ret2)  
- Tunnel : PDR = 0.368 (ret0) → 0.489 (ret1) → 0.542 (ret2)

Cela indique qu’en l’absence de contention, le goulot est la réussite de propagation ; la retransmission convertit une partie des échecs PHY en succès à temps.

**Observation 2 : Cong — effondrement global et « différences de scénario au second ordre »**  
Sous Cong, les PDR sont comprimés et les écarts entre scénarios se réduisent. Fig07 (dist≤200 m) :

- Ref : PDR = 0.054 (ret0) → 0.090 (ret1) → 0.117 (ret2)  
- UrbMask : PDR = 0.057 (ret0) → 0.097 (ret1) → 0.126 (ret2)  
- Tunnel : PDR = 0.049 (ret0) → 0.082 (ret1) → 0.107 (ret2)

Le goulot bascule vers **contention MAC / collisions / file d’attente** ; les différences de propagation deviennent secondaires, ce que Fig04 étaye.

**Phrase de conclusion**  
NoCong : ret améliore nettement la fiabilité et révèle les différences de propagation ; Cong : effondrement global et convergence inter-scénarios → la congestion domine.

---

## Fig02 : Délais de queue (p95/p99) vs distance (NoCong/Cong ; ret=0 vs ret=2)

![](assets/final_figures_A_preview/Fig02_TailDelay_p95p99_Ret0Ret2.png)

**Sens et rôle**  
Fig02 mesure le coût : la hausse de ret épaissit-elle la queue de délai ? La congestion rapproche-t-elle la queue de la deadline, activant la pénalité de temporalité ?

**NoCong (haut) : queue accrue mais loin de la deadline (late≈0)**  
Sous NoCong, p95 est faible à ret0, et ret2 élève p95/p99 vers 10–30 ms. Fig07 (dist≤200 m) :

- Ref : p95=2.9 ms (ret0) → 22.4 ms (ret2)  
- UrbMask : p95=3.9 ms (ret0) → 22.9 ms (ret2)  
- Tunnel : p95=3.9 ms (ret0) → 22.9 ms (ret2)

En parallèle, Fig07 indique late=0.0% sous NoCong : la queue augmente mais reste sous la deadline.

**Cong (bas) : queue portée à 70–100 ms ; la deadline devient structurante**  
Sous Cong, p95 ≈ 70–85 ms et p99 ≈ 90–100 ms. Fig07 :

- Ref : p95=69.6 ms (ret0) → 79.9 ms (ret2)  
- UrbMask : p95=70.1 ms (ret0) → 79.9 ms (ret2)  
- Tunnel : p95=69.1 ms (ret0) → 79.6 ms (ret2)

Ainsi, Cong élève la queue vers l’échelle de la deadline ; ret augmente encore la queue et fait apparaître des « succès PHY mais tardifs », mécanisme décomposé par Fig03.

**Phrase de conclusion**  
NoCong : coût = hausse de queue acceptable ; Cong : queue proche de la deadline → la pénalité late devient une composante majeure.

---

## Fig03 : Décomposition Cong-only (succès PHY, succès à temps, pénalité late)

![](assets/final_figures_A_preview/Fig03_Cong_Decomposition_3Curves.png)

**Sens et rôle**  
Fig03 explique pourquoi le timely PDR est faible sous Cong : pertes dues au PHY ou aux late ? Comment ret se traduit-il en gain/coût ?

**Lecture (à expliciter dans le rapport)**  
Axe gauche :

- Bleu : \(\mathrm{PDR}*{\mathrm{phy}}=N*{\mathrm{success_phy}}/N_{\mathrm{total}}\)  
- Vert : \(\mathrm{PDR}*{\mathrm{timely}}=N*{\mathrm{success}}/N_{\mathrm{total}}\)

Axe droit (orange pointillé) :

- \(\mathrm{late_ratio}*{\mathrm{phy}}=N*{\mathrm{late}}/N_{\mathrm{success_phy}}\)

avec :
$$
\mathrm{PDR}*{\mathrm{timely}}=\mathrm{PDR}*{\mathrm{phy}}\cdot(1-\mathrm{late_ratio}_{\mathrm{phy}})
$$

**Observation 1 : Cong — terme dominant initial = faible succès PHY**  
Sur 0–200 m, la courbe bleue est déjà faible, cohérente avec p_col élevé et délais de congestion de Fig04 : contention et collisions causent des échecs PHY.

**Observation 2 : avec la distance, late_ratio augmente et écarte davantage timely de PHY**  
La hausse de late_ratio élargit l’écart vert/bleu, en cohérence avec Fig02 : la file/backoff pousse la queue, convertissant des succès PHY en paquets tardifs.

**Observation 3 : ret=2 est un “double tranchant” quantifiable**  
ret augmente phy_rate mais peut aussi augmenter late_ratio. Fig07 quantifie late% :

- Ref : late=5.5% (ret0) → 8.7% (ret2)  
- UrbMask : late=5.4% (ret0) → 8.4% (ret2)  
- Tunnel : late=5.1% (ret0) → 7.9% (ret2)

Le gain net est donc la compétition entre gain PHY et pénalité late, sous une structure multiplicative.

**Phrase de conclusion**  
Sous Cong, la baisse du timely PDR se décompose strictement en baisse du succès PHY et hausse de la pénalité late ; la retransmission augmente les deux, avec des bornes quantifiables.

---

## Fig04 : Preuves proxy de congestion (moyenne ± écart-type)

![](assets/final_figures_A_preview/Fig04_CongProxy_MeanStdBand.png)

**Sens et rôle**  
Fig04 étaye la convergence inter-scénarios sous Cong : les proxys de congestion sont-ils similaires entre scénarios ?

**Observation : proxys très cohérents, bandes de variance étroites**  
CBR moyen, p_col moyen et délai de congestion moyen ont des formes proches et des bandes ±1 std étroites sur dist≤200 m. Cong impose une intensité de canal occupé comparable, rendant la congestion dominante et la propagation secondaire, ce qui explique Fig01.

**Phrase de conclusion**  
Les proxys de congestion sont cohérents entre scénarios, soutenant la thèse de « congestion dominante ».

---

## Fig05 : Hétérogénéité spatiale UrbMask (hétérogénéité + amplification par congestion)

![](assets/final_figures_A_preview/Fig05_UrbMask_Heterogeneity_LinesAndRatio.png)

**Sens et rôle**  
Fig05 examine l’hétérogénéité spatiale : le masquage urbain est-il fortement dépendant de la position ? La congestion amplifie-t-elle les zones faibles ?

**Haut (NoCong) : PDR_band (80–100 m) varie avec mid_x**  
Dans la bande 80–100 m, PDR_band présente une structure spatiale continue. ret=2 augmente le niveau global mais conserve la texture : la non-uniformité demeure.

**Bas (ratio=Cong/NoCong) : pénalisation accrue des zones faibles**  
Le ratio <1 partout, avec des creux plus marqués dans certaines zones. Cela indique une amplification : la congestion pénalise davantage les régions déjà faibles, poussant localement vers l’indisponibilité.

**Phrase de conclusion**  
UrbMask présente une forte hétérogénéité spatiale ; la congestion amplifie les zones faibles via une baisse du ratio Cong/NoCong.

---

## Fig06 : Effet segmenté Tunnel (inside vs outside)

![](assets/final_figures_A_preview/Fig06_Tunnel_InsideOutside_Bars_UnifiedY.png)

**Sens et rôle**  
Fig06 vérifie la segmentation inside/outside : inside est-il durablement plus faible ? Cong rend-il inside quasi inutilisable ? ret et late y sont-ils quantifiables ?

**NoCong (haut) : inside << outside ; ret améliore les deux**  
Barres = timely PDR (80–100 m), labels = PHY et late (NoCong late≈0). Valeurs typiques :

- ret0 : inside phy=0.115, outside phy=0.182  
- ret1 : inside phy=0.214, outside phy=0.333  
- ret2 : inside phy=0.304, outside phy=0.451  

L’écart persiste : segmentation structurée.

**Cong (bas) : inside quasi inutilisable, late>0 ; ret augmente aussi le risque de retard**  
Valeurs typiques :

- ret0 : inside phy=0.015, late=0.059 ; outside phy=0.034, late=0.030  
- ret2 : inside phy=0.043, late=0.076 ; outside phy=0.097, late=0.044  

Cong dégrade inside par baisse PHY et hausse de retard, cohérent avec Fig02/Fig03.

**Phrase de conclusion**  
Segmentation tunnel stable ; Cong pousse inside vers l’indisponibilité et introduit des pénalités late ; ret améliore PHY mais accroît le risque de retard.

---

## Fig07 : Matrices de synthèse (dist≤200 m)

![](assets/final_figures_A_preview/Fig07_Summary_Matrices.png)

**Sens et rôle**  
Fig07 fournit, pour chaque (scénario × ret × régime), le triplet **PDR, p95, late%** (dist≤200 m). Elle sert d’ancrage numérique et de synthèse d’arbitrage.

**NoCong : gain PDR fort, coût p95 clair, late≈0**

- Ref : PDR 0.439→0.582 ; p95 2.9→22.4 ms ; late 0.0%  
- UrbMask : PDR 0.462→0.644 ; p95 3.9→22.9 ms ; late 0.0%  
- Tunnel : PDR 0.368→0.542 ; p95 3.9→22.9 ms ; late 0.0%  

**Cong : PDR faible, p95 proche deadline, late% augmente avec ret**

- Ref : PDR 0.054→0.117 ; p95 69.6→79.9 ms ; late 5.5→8.7%  
- UrbMask : PDR 0.057→0.126 ; p95 70.1→79.9 ms ; late 5.4→8.4%  
- Tunnel : PDR 0.049→0.107 ; p95 69.1→79.6 ms ; late 5.1→7.9%  

**Phrase de conclusion**  
NoCong : ret augmente fortement PDR au prix d’une queue plus épaisse ; Cong : la congestion écrase PDR et ret augmente late% → co-conception avec contrôle de congestion.

---

# Synthèse inter-figures (fermeture “phénomène–mécanisme–implication”)

1. **Basculement de goulot** : NoCong → propagation ; Cong → contention/collision/file d’attente (Fig01 + Fig04).  
2. **Frontière gain–coût** : NoCong → gains PDR + coûts de queue ; Cong → gains plus faibles + hausse late% (Fig02/03/07).  
3. **Dégradation structurée** : UrbMask → hétérogénéité spatiale amplifiée par congestion (Fig05) ; Tunnel → segmentation inside/outside, Cong extrémise inside (Fig06).  
4. **Implication** : la retransmission seule ne suffit pas ; il faut une co-conception avec contrôle de congestion pour maîtriser phy_rate et late_ratio (cadre de Fig03).

---

# Limites et formulations recommandées (rigueur)

1. **Résolution et lissage** : agrégation par bins + lissage léger (visualisation), sans modifier les comptages/histogrammes ; conclusions numériques sur Fig06/Fig07.  
2. **Quantiles** : p95/p99 estimés via histogrammes des succès à temps ; masquage lorsque l’échantillonnage est insuffisant.  
3. **Rôle des proxys** : indicateurs internes contrôlables pour étayer « congestion dominante », sans prétendre reproduire bit-à-bit une pile normalisée ; valeur = contraste explicable et discussion de limites.


## Aperçu de la structure du code (repérage rapide)

Le code se trouve dans `03_sim_A/py/` et `03_sim_A/py/modules/`. La chaîne est orchestrée par `run_pipeline_A.py` : génération d’entrées → simulation → agrégation → figures → audit.

Une description plus détaillée « fichier par fichier (fonctions/classes clés et points d’entrée de paramétrage) » est fournie dans le Notebook 2.
